In [3056]:
import pandas as pd
df = pd.read_excel(r'D:\PYTHON\Machine Learning\Data_set\Machine_Failure_Prediction_Custom_Dataset_Messy.xlsx')

In [3057]:
df = df.drop_duplicates()
df.columns = df.columns.str.strip().str.lower()
print(df.shape)
print(df.info())
print(df.head(5))
print(df.isnull().sum())

(505, 12)
<class 'pandas.DataFrame'>
RangeIndex: 505 entries, 0 to 504
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   machine_id            505 non-null    str    
 1   machine_type          505 non-null    str    
 2   temperature_c         493 non-null    float64
 3   vibration_mm_s        505 non-null    float64
 4   pressure_bar          505 non-null    float64
 5   motor_current_a       505 non-null    float64
 6   power_kw              505 non-null    float64
 7   operating_hours       505 non-null    int64  
 8   oil_level_pct         482 non-null    float64
 9   maintenance_history   505 non-null    str    
 10  ambient_humidity_pct  505 non-null    int64  
 11  machine_status        505 non-null    str    
dtypes: float64(6), int64(2), str(4)
memory usage: 47.5 KB
None
  machine_id machine_type  temperature_c  vibration_mm_s  pressure_bar  \
0      M1002      Milling           54.4    

In [3058]:
import numpy as np

num_col = df.select_dtypes(include='number').columns
df[num_col] = df[num_col].mask(df[num_col]<0, np.nan)

In [3059]:
df['machine_type'] = df['machine_type'].str.strip().str.lower()
df['maintenance_history'] = df['maintenance_history'].str.strip().str.lower()
df['machine_status'] = df['machine_status'].str.strip().str.lower()

In [3060]:
print(df['machine_type'].unique())
print(df['machine_type'].nunique())
print(df['maintenance_history'].unique())
print(df['maintenance_history'].nunique())

<StringArray>
['milling', 'lathe', 'drill', 'press', 'cnc']
Length: 5, dtype: str
5
<StringArray>
['no', 'yes']
Length: 2, dtype: str
2


In [3061]:
from sklearn.model_selection import train_test_split

unnecessary_col = ['machine_id','machine_status']

X = df.drop(unnecessary_col, axis=1)
y = df['machine_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.reset_index(drop=True))
print(X_test.reset_index(drop=True))
print(y_train.reset_index(drop=True))
print(y_test.reset_index(drop=True))

    machine_type  temperature_c  vibration_mm_s  pressure_bar  \
0          lathe           64.4           10.22           4.3   
1          press           53.9            6.77           2.8   
2          drill           92.5            4.48           9.4   
3          press           51.1            2.38           5.6   
4          drill           66.7            3.12           7.7   
..           ...            ...             ...           ...   
399        lathe           83.1            9.10           8.0   
400        drill           92.7            9.05           4.7   
401        drill           86.3            9.27           2.6   
402        drill           52.2            5.73           8.7   
403        press           51.6            8.53           6.0   

     motor_current_a  power_kw  operating_hours  oil_level_pct  \
0               25.4       2.9            11253           86.0   
1               15.9       3.9            18670           38.0   
2               38.7 

In [3062]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

fill_col = ['temperature_c', 'oil_level_pct']

X_train[fill_col] = imputer.fit_transform(X_train[fill_col])
X_test[fill_col] = imputer.transform(X_test[fill_col])

In [3063]:
print(X_train.isnull().sum().sum())
print(X_test.isnull().sum().sum())

0
0


In [3064]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

X_train['maintenance_history'] = le.fit_transform(X_train['maintenance_history'])
X_test['maintenance_history'] = le.transform(X_test['maintenance_history'])

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [3065]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[['machine_type']])
X_test_encoded = encoder.transform(X_test[['machine_type']])

In [3066]:
X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns = encoder.get_feature_names_out(['machine_type']),
    index = X_train.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns= encoder.get_feature_names_out(['machine_type']),
    index = X_test.index
)

In [3067]:
X_train = X_train.drop(['machine_type'], axis=1)
X_test = X_test.drop(['machine_type'], axis=1)

X_train = pd.concat([X_train, X_train_encoded], axis=1)
X_test = pd.concat([X_test, X_test_encoded], axis=1)

In [3068]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [3069]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

print(le.inverse_transform(y_pred)[0])

normal


In [3070]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, f1_score, classification_report

print("Accuracy Score :", accuracy_score(y_test, y_pred))
print("Precision Score :", precision_score(y_test, y_pred))
print("Recall Score :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test, y_pred))
print('Confusion Matrix :')
print(confusion_matrix(y_test, y_pred))

Accuracy Score : 0.8811881188118812
Precision Score : 0.8857142857142857
Recall Score : 0.9393939393939394
F1 Score : 0.9117647058823529
Classification Report :
              precision    recall  f1-score   support

           0       0.87      0.77      0.82        35
           1       0.89      0.94      0.91        66

    accuracy                           0.88       101
   macro avg       0.88      0.86      0.86       101
weighted avg       0.88      0.88      0.88       101

Confusion Matrix :
[[27  8]
 [ 4 62]]


In [3071]:
from sklearn.inspection import permutation_importance

result = permutation_importance(model, X_test_scaled, y_test)
for feature, importance in zip(X_train.columns, result.importances_mean):
    print(feature, ":", importance)

temperature_c : 0.14851485148514854
vibration_mm_s : 0.057425742574257456
pressure_bar : 0.0
motor_current_a : 0.047524752475247546
power_kw : 0.0
operating_hours : 0.08712871287128712
oil_level_pct : 0.003960396039603964
maintenance_history : 0.09306930693069307
ambient_humidity_pct : 0.001980198019801982
machine_type_cnc : -0.001980198019801982
machine_type_drill : 0.0
machine_type_lathe : 0.0
machine_type_milling : 0.0
machine_type_press : 0.011881188118811892


In [3072]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=5)

model.fit(X_train_scaled, y_train)
y_pred_knn = model.predict(X_test_scaled)
print(le.inverse_transform(y_pred_knn)[0])

normal


In [3073]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_knn))
print("Precision :", precision_score(y_test, y_pred_knn))
print("Recall :", recall_score(y_test, y_pred_knn))
print("Confusion Matrix :")
print(confusion_matrix(y_test, y_pred_knn))

Accuracy : 0.7722772277227723
Precision : 0.7654320987654321
Recall : 0.9393939393939394
Confusion Matrix :
[[16 19]
 [ 4 62]]


In [3074]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

model.fit(X_train_scaled, y_train)

y_pred_rfc = model.predict(X_test_scaled)

print(le.inverse_transform(y_pred_rfc)[0])

normal


In [3075]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

print("Accuracy :", accuracy_score(y_test, y_pred_rfc))
print("Precision :", precision_score(y_test, y_pred_rfc))
print("Recall :", recall_score(y_test, y_pred_rfc))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rfc))

Accuracy : 0.9207920792079208
Precision : 0.9027777777777778
Recall : 0.9848484848484849

Confusion Matrix:
[[28  7]
 [ 1 65]]


In [3076]:
print("Conclusion: Random Forest Classifier achieved the best performance and was selected as the final model.")

Conclusion: Random Forest Classifier achieved the best performance and was selected as the final model.


In [3077]:
import joblib

joblib.dump(model, "machine_failure_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(encoder, "onehot_encoder.pkl")
joblib.dump(le, "label_encoder.pkl")

print("All files saved successfully!")

All files saved successfully!
